# 1. Проекты, обучение, экзамены и настройки
Чистовая версия не импортирует data_collection/. Старые файлы сохранены. По умолчанию все API-записи выключены. Не выполняйте платные этапы через «Run All»: запуск каждого пула вынесен отдельно.

`existing` подключает текущие проекты; `new` создаёт самостоятельное воспроизведение с новым namespace. Для проверки архива без токена используйте ноутбук 04. Токен — скрытый ввод или переменная YANDEX_TASKS_TOKEN, в файлы не записывается.

In [ ]:
from pathlib import Path
import sys, os, getpass, importlib
ROOT = Path.cwd()
if ROOT.name == 'clean_annotation':
    ROOT = ROOT.parent
CLEAN = ROOT / 'clean_annotation'
if not (CLEAN / 'workflow.py').exists():
    raise RuntimeError('Откройте ноутбук из корня проекта или clean_annotation')
if str(CLEAN) not in sys.path:
    sys.path.insert(0, str(CLEAN))
import workflow as w
importlib.reload(w)
PROFILE = 'existing'  # existing: текущие проекты; new: независимое воспроизведение
ENABLE_WRITES = False  # явное включение создания/изменений; запуск заданий отдельными ячейками
TOKEN = os.environ.get('YANDEX_TASKS_TOKEN') or getpass.getpass('OAuth-токен (скрытый ввод): ')
api = w.Client(TOKEN, write=ENABLE_WRITES)
del TOKEN


## Единый файл настроек
Откройте config.json: инструкции, цены за СТРАНИЦУ, фото на странице, перекрытие, кандидаты экзамена и сроки. Exam overlap — число кандидатов; все 10 разных примеров выдаются одной страницей. BBOX training: 5 примеров, REVIEW training: 6; контрольные REVIEW: 27. Экзамен REVIEW автоматический; BBOX — отложенная приёмка 1 день и оценка качества боксов через REVIEW. Основной набор: пилот 25 + main 375, без пересечений.

Цены по умолчанию повторяют согласованные низкие ставки; доступный минимум дополнительно проверяет сервер. Изменение JSON само по себе НЕ меняет живые пулы.

In [ ]:
w.config()

In [ ]:
w.plan_costs()

## Подключить существующие проекты
Только чтение. Для другого аккаунта выберите new. Отказ проверки владельца означает неверный аккаунт — не меняйте IDs наугад.

In [ ]:
if PROFILE == 'existing':
    w.attach_existing(api)
    display(w.inventory(api, PROFILE))

## Создать новый закрытый набор
Перед запуском выберите PROFILE=new, измените namespace и expires_at в config.json. Затем включите ENABLE_WRITES=True в первой ячейке и повторите её. Новые проекты, навыки, обучения и пулы создаются закрытыми, задания загружаются. Повторный запуск восстанавливает реестр по маркерам, а не создаёт дубликаты. При неясном исходе сетевого запроса код останавливается для сверки.

In [ ]:
if PROFILE == 'new' and ENABLE_WRITES:
    display(w.prepare(api, PROFILE))
else:
    print('Создание выключено или выбран existing.')

## Поменять отдельную цену или расширить экзамен
Ресурсы не пересоздаются. Укажите ключ существующего этапа. Изменение перекрытия обновит уже загруженные конечные задания; повторное указание 40 не добавит ещё 20. Для обоих экзаменов сохраняйте 10 примеров на странице. Цена уже взятой страницы может остаться прежней.

In [ ]:
APPLY_UPDATE = False
STAGE = 'review_main'
CHANGES = {'price': 0.15}
# Для добора: STAGE='review_exam'; CHANGES={'overlap':40}
if APPLY_UPDATE:
    display(w.update_settings(api, PROFILE, STAGE, **CHANGES))

## Применить пороги REVIEW
Измените review_exam_pass и review_control_pass в config.json и вызовите эту ячейку. Она обновит фильтры и правила контроля существующих пулов, сохраняя цены. Уже выданные блокировки не исчезают от смены порога: список будет возвращён для проверки причины.

In [ ]:
APPLY_QUALITY = False
if APPLY_QUALITY:
    display(w.update_quality(api, PROFILE))
